# 03 — Tools de verdad

**Bloque 3, 25 minutos.** El código vive en
[`src/paso3_tools_http/agente.py`](../src/paso3_tools_http/agente.py).

Son las mismas tools del paso 2, pero ahora llaman a un backend por HTTP que
escribe en un archivo que puedes abrir y mirar.

**Este notebook habla con dos máquinas: Ollama y el backend.** La primera celda
carga el `.env` de la raíz, comprueba las dos y dice a qué dirección va cada
una. Si el backend no está arriba, levántalo con
`scripts/<tu-sistema>/taller-up`.

Corre el notebook de arriba abajo. Tarda entre 40 y 60 segundos contra el
servidor del Tec: tres corridas el 2026-11-15 dieron 42, 42 y 57. Una de las
celdas apaga el backend a propósito antes de volver a levantarlo.

> **Tu salida no va a ser idéntica a la guardada, y eso es parte del tema.** En
> dos corridas del mismo día, con el mismo inventario de partida y temperatura
> 0, el agente tomó caminos distintos en dos celdas. Donde pasa está marcado, y
> la sección 2 lo mide.

> **Si la primera celda avisa que quitó claves del entorno, es lo esperado.** El
> `.env` de la raíz manda, y lo que un kernel viejo trajera se descarta. Por qué
> hizo falta está en [`planeacion/hallazgos.md`](../planeacion/hallazgos.md).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env, modelo_configurado, origen_url

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")

import paso3_tools_http.agente as p3
importlib.reload(p3)   # vuelve a correr esto después de editar el archivo

import requests

# Este notebook habla con DOS maquinas, y las dos se comprueban aqui. Sin esto,
# un Ollama mal apuntado se descubre cinco celdas mas abajo.
_url, _motivo = origen_url()
print(f"  ollama : {_url}    modelo: {modelo_configurado()}")
print(f"           lo decidió: {_motivo}")
requests.get(f"{_url}/api/version", timeout=10).raise_for_status()

_salud = requests.get(f"{p3.BACKEND}/api/health", timeout=5).json()
print(f"  backend: {p3.BACKEND}    salud: {_salud}")

  .env   : c:\Users\Mario\Documents\desarrollo\tec\agenticai\.env  (19 claves)
  ollama : http://10.49.12.54:11434    modelo: qwen3.5:9b
           lo decidió: OLLAMA_DESTINO=tec, en el .env de la raíz
  backend: http://localhost:8000    salud: {'estado': 'ok', 'trazas': 0}


## Lo único que cambió

Abre los dos archivos uno al lado del otro:

- [`src/paso2_herramienta/agente.py`](../src/paso2_herramienta/agente.py)
- [`src/paso3_tools_http/agente.py`](../src/paso3_tools_http/agente.py)

La celda siguiente compara el **bucle** de los dos, carácter por carácter, y
después imprime la línea que sí cambió en cada una de las dos tools.

In [2]:
import inspect

import paso2_herramienta.agente as p2

a = inspect.getsource(p2.correr)
b = inspect.getsource(p3.correr)

print(f"  correr() del paso 2: {len(a)} caracteres")
print(f"  correr() del paso 3: {len(b)} caracteres")
print(f"  ¿idénticos?          {'SÍ' if a == b else 'NO'}")

print("\n  Y las funciones que SÍ cambiaron:")
print("    paso 2:", inspect.getsource(p2.consultar_stock).strip().split("\n")[1].strip()[:60])
print("    paso 3:", inspect.getsource(p3.consultar_stock).strip().split("\n")[-1].strip()[:60])

  correr() del paso 2: 2108 caracteres
  correr() del paso 3: 2108 caracteres
  ¿idénticos?          SÍ

  Y las funciones que SÍ cambiaron:
    paso 2: producto = INVENTARIO.get(sku)
    paso 3: return _pedir("GET", f"/api/inventario/{sku}")


**Ese es el punto del paso 3.** Cambió el cuerpo de dos funciones: donde había
`INVENTARIO.get(sku)` ahora hay un `requests.get()`. El bucle no se tocó, el
esquema de las tools no se tocó y el modelo no recibió nada distinto.

Una tool es un **contrato**: un nombre, unos parámetros y un resultado. Lo que
pase del otro lado es asunto tuyo. Hoy es un CSV; mañana puede ser una base de
datos, un ERP o una llamada a otro equipo.

> El paso 3 agrega además `actualizar_existencias`, que escribe, y
> `_sin_existencias`, que recorta la respuesta del catálogo antes de dársela al
> modelo. Lo segundo parece un detalle y decide casi todo lo que hace el agente
> de aquí en adelante. La sección 5 lo mide.

## Lo que hay en el sistema, antes de preguntar

La celda siguiente imprime el inventario, que ahora sale del backend y no de un
diccionario en memoria.

Ten [`datos/inventario.csv`](../datos/inventario.csv) abierto en otra pestaña
durante todo el notebook: las celdas de escritura lo van a cambiar, y verlo
cambiar es medio bloque.

In [3]:
def tabla():
    d = requests.get(f"{p3.BACKEND}/api/inventario", timeout=10).json()
    print(f"  {'SKU':<15} {'producto':<30} {'exist.':>7} {'precio':>7}  categoría")
    print(f"  {'-'*15} {'-'*30} {'-'*7} {'-'*7}  {'-'*11}")
    for p in d["productos"]:
        aviso = "  <- en cero" if p["existencias"] == 0 else ""
        print(f"  {p['sku']:<15} {p['nombre']:<30} {p['existencias']:>7} "
              f"{p['precio']:>7.0f}  {p['categoria']}{aviso}")
    print(f"\n  {d['total']} productos")

tabla()

  SKU             producto                        exist.  precio  categoría
  --------------- ------------------------------ ------- -------  -----------
  CAFE-500        Café molido 500g                    17     180  cafe
  CAFE-250        Café molido 250g                    34      95  cafe
  CAFE-GRANO-1K   Café en grano 1kg                    8     340  cafe
  TE-200          Té verde 200g                        0      95  te  <- en cero
  TE-MANZ-100     Té de manzanilla 100g               23      62  te
  AZUCAR-1K       Azúcar 1kg                          42      38  endulzante
  MIEL-500        Miel de abeja 500g                  11     120  endulzante
  LECHE-1L        Leche entera 1L                      3      29  lacteo
  LECHE-DESL-1L   Leche deslactosada 1L               19      34  lacteo
  VASO-360        Vaso desechable 360ml (50 pz)        7      85  desechable

  10 productos


## 1. Leer

Las dos celdas siguientes hacen las mismas preguntas del paso 2, ahora contra el
backend. Hay dos cosas nuevas que mirar.

**El catálogo del paso 3 acepta una categoría, y el modelo la usa.** Pide
`consultar_catalogo({'categoria': 'cafe'})` en vez de traerse los diez
productos. Ese parámetro no existía en el paso 2.

**Y el catálogo no devuelve existencias:** da clave, nombre, precio y categoría.
Para un número de unidades hace falta `consultar_stock`. Fíjate en qué hace cada
pregunta cuando necesita un número y el modelo solo tiene nombres.

In [4]:
p3.correr("¿Cuánto café tenemos?")


  --- vuelta 1 ---
  pide: consultar_catalogo({'categoria': 'cafe'})
  resultado: {'productos': [{'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'precio': 180.0, 'categoria': 'cafe'}, {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'precio': 95.0, 'categoria': 'cafe'}, {'sku': 'CAFE-GRANO-1K', 'nombre': 'Café en grano 1kg', 'precio': 340.0, 'categoria': 'cafe'}], 'total': 3}

  --- vuelta 2 ---
  responde: Tengo tres tipos de café: CAFE-500, CAFE-250 y CAFE-GRANO-1K. ¿Podrías decirme cuál te interesa para saber cuántas unidades tenemos?


'Tengo tres tipos de café: CAFE-500, CAFE-250 y CAFE-GRANO-1K. ¿Podrías decirme cuál te interesa para saber cuántas unidades tenemos?'

In [5]:
# Existencias en cero. No es lo mismo que "no existe".
p3.correr("¿Hay té verde disponible?")


  --- vuelta 1 ---
  pide: consultar_catalogo({'categoria': 'te'})
  resultado: {'productos': [{'sku': 'TE-200', 'nombre': 'Té verde 200g', 'precio': 95.0, 'categoria': 'te'}, {'sku': 'TE-MANZ-100', 'nombre': 'Té de manzanilla 100g', 'precio': 62.0, 'categoria': 'te'}], 'total': 2}

  --- vuelta 2 ---
  responde: Sí, hay té verde disponible (SKU: TE-200). ¿Deseas saber cuántas unidades hay en stock?


'Sí, hay té verde disponible (SKU: TE-200). ¿Deseas saber cuántas unidades hay en stock?'

### Las dos lecturas

| pregunta | pidió | contestó |
|---|---|---|
| "¿Cuánto café tenemos?" | el catálogo de `cafe`, y después los tres `consultar_stock` | los tres, con 17, 34 y 8 unidades |
| "¿Hay té verde disponible?" | el catálogo de `te`, y nada más | *"Sí, hay té verde disponible (SKU: TE-200)"* |

La primera encadenó dos tools porque el número no estaba en el catálogo. Son
tres vueltas y cuatro llamadas, y las tres de `consultar_stock` caen en la misma
vuelta.

**La segunda respuesta no es cierta en el sentido que importa.** El `TE-200`
tiene 0 unidades, y el modelo no las consultó. Contestó con el dato que sí
tenía, que el producto existe en el catálogo, y cerró ofreciendo mirar el stock.

Confundir "existe" con "hay" es un error de diseño tanto como del modelo. **El
catálogo que le dimos no distingue las dos cosas**, así que la distinción tiene
que salir de una segunda llamada que nada lo obliga a hacer. La sección 5 lo
mide.

> **La pregunta del café varía entre corridas.** En otra del mismo día contestó
> en dos vueltas y una sola llamada: listó los tres SKU y preguntó cuál te
> interesa, sin consultar ningún stock. Esa también es una buena respuesta, y es
> otra.

## 2. Escribir

Hasta aquí el agente solo miraba. `actualizar_existencias` **modifica el
sistema**: después de esta celda, el archivo en disco es distinto.

Mira `datos/inventario.csv` antes y después de correrla.

In [6]:
antes = requests.get(f"{p3.BACKEND}/api/inventario/LECHE-1L", timeout=10).json()
print("ANTES :", antes, "\n")

p3.correr("Acaban de llegar 50 unidades más de leche entera, actualízalo")

despues = requests.get(f"{p3.BACKEND}/api/inventario/LECHE-1L", timeout=10).json()
print("\nDESPUÉS:", despues)

ANTES : {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'existencias': 3, 'precio': 29.0, 'categoria': 'lacteo'} 


  --- vuelta 1 ---
  pide: consultar_catalogo({'categoria': 'lacteo'})
  resultado: {'productos': [{'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'precio': 29.0, 'categoria': 'lacteo'}, {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'precio': 34.0, 'categoria': 'lacteo'}], 'total': 2}

  --- vuelta 2 ---
  pide: consultar_stock({'sku': 'LECHE-1L'})
  resultado: {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'existencias': 3, 'precio': 29.0, 'categoria': 'lacteo'}

  --- vuelta 3 ---
  pide: actualizar_existencias({'sku': 'LECHE-1L', 'existencias': 53})
  resultado: {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'existencias': 53, 'precio': 29.0, 'categoria': 'lacteo'}

  --- vuelta 4 ---
  responde: ¡He actualizado las existencias de leche entera a 53 unidades.

DESPUÉS: {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'existencias': 53, 'precio': 29.0

### Lo que acaba de pasar

Tres vueltas y dos tools:

| vuelta | pidió |
|---|---|
| 1 | `consultar_catalogo({'categoria': 'lacteo'})`, porque la petición dice "leche entera" y no un SKU |
| 2 | `actualizar_existencias({'sku': 'LECHE-1L', 'existencias': 50})` |
| 3 | nada: responde |

**El producto tenía 3 unidades y quedó en 50.** La petición decía que llegaron
50 *más*, así que lo correcto era 53. El modelo escribió el número de la
petición sin sumarlo al que había, y nunca pidió el que había.

En un inventario real eso no es un error de redondeo. Son tres unidades
perdidas, y **el archivo queda afirmando que la cuenta cuadró**: el backend
aceptó el 50 sin protestar, porque 50 es un entero válido y el backend no sabe
qué pediste.

### La misma celda, en otra corrida

Con el mismo inventario de partida, el mismo servidor y la misma temperatura 0,
esta celda dio dos caminos distintos el 2026-11-15:

| | vueltas | tools | LECHE-1L quedó en |
|---|---|---|---|
| una corrida | 4 | catálogo, `consultar_stock`, `actualizar_existencias` | **53**, correcto |
| la otra | 3 | catálogo, `actualizar_existencias` | **50**, incorrecto |

Toda la diferencia es si pidió `consultar_stock` antes de escribir. **Cuando lo
pide, suma bien; cuando no, sobrescribe.** Nada en la petición decide cuál de
los dos va a hacer.

> **Temperatura 0 no es determinismo.** Lo es para una respuesta corta —1 de 8
> salidas distintas, medido en
> [`hallazgos.md`](../planeacion/hallazgos.md)— y deja de serlo cuando la
> conversación lleva varios resultados de tools encima. Corre la celda otra vez
> y compara.

**Eso es el argumento del paso 4, sin necesidad de más.** Una tool que escribe
sin leer primero puede corromper datos, el orden en que el modelo las pide no
está garantizado, y un system prompt que diga "consulta antes de escribir" es
una sugerencia. La garantía se escribe en código.

> **Y el error como dato no aparece en esta corrida.** Aquí el modelo sacó la
> clave del catálogo y no se equivocó. Las secciones 3 y 4 le dan un rechazo del
> backend y una caída de red, y en las dos el resultado vuelve como un
> diccionario con `error` en vez de reventar el programa. Si `consultar_stock`
> lanzara una excepción de Python, el agente se caería.

## 3. Cuando el sistema dice que no

El backend valida: `existencias` no puede ser negativo. La celda siguiente pide
una resta imposible, para ver cómo se entera el modelo y qué hace con el
rechazo.

In [7]:
p3.correr("Pon el stock de CAFE-500 en -5")


  --- vuelta 1 ---
  pide: actualizar_existencias({'sku': 'CAFE-500', 'existencias': -5})
  resultado: {'error': [{'type': 'greater_than_equal', 'loc': ['body', 'existencias'], 'msg': 'Input should be greater than or equal to 0', 'input': -5, 'ctx': {'ge': 0}}]}

  --- vuelta 2 ---
  responde: No puedo poner el stock en -5. El sistema no permite existencias negativas. ¿Quieres ajustar el stock a un número positivo?


'No puedo poner el stock en -5. El sistema no permite existencias negativas. ¿Quieres ajustar el stock a un número positivo?'

## 4. Cuando el sistema no está

La celda siguiente **apaga el backend a propósito**, le pregunta al agente y
después lo vuelve a levantar. Tarda unos 13 segundos, medidos el 2026-11-15.

Vale la pena correrla. Tarde o temprano vas a tener el backend caído sin
saberlo, y conviene reconocer el síntoma: el agente no se rompe, reporta un
error de conexión que viene de `requests` y ofrece otra cosa.

In [8]:
import subprocess

RAIZ = Path.cwd().parent
subprocess.run(["docker", "compose", "stop", "backend"],
               cwd=RAIZ / "infra" / "taller", capture_output=True)
print("backend detenido\n")

p3.correr("¿Cuántas unidades hay de CAFE-500?")

subprocess.run(["docker", "compose", "start", "backend"],
               cwd=RAIZ / "infra" / "taller", capture_output=True)
import time
time.sleep(6)
print("\nbackend de vuelta:",
      requests.get(f"{p3.BACKEND}/api/health", timeout=10).json())

backend detenido


  --- vuelta 1 ---
  pide: consultar_stock({'sku': 'CAFE-500'})
  resultado: {'error': 'no pude hablar con el backend en http://localhost:8000', 'sugerencia': 'esta levantado? -> scripts/<tu-sistema>/taller-up'}

  --- vuelta 2 ---
  responde: Lo siento, no puedo obtener la información del stock en este momento porque hay un problema de conexión con el sistema. Por favor, inténtalo más tarde o contáctame cuando el sistema esté operativo nue

backend de vuelta: {'estado': 'ok', 'trazas': 0}


## 5. Cuántas vueltas da el bucle

Lo que lleva el bucle en esta corrida, celda por celda:

| pregunta | vueltas | tools |
|---|---|---|
| "¿Cuánto café tenemos?" | 3 | 4 |
| "¿Hay té verde disponible?" | 2 | 1 |
| "llegaron 50 más de leche entera" | 3 | 2 |
| "pon CAFE-500 en -5" | 2 | 1 |
| con el backend caído | 2 | 1 |

**Las de dos vueltas son las que se contestan con una sola llamada:** el té
verde salió del catálogo, la escritura inválida murió en el rechazo y la del
backend caído murió en el error. Pedir una tool y responder son dos vueltas, y
ese es el piso.

**Lo que alarga la cadena es la dependencia secuencial:** que el modelo no pueda
saber qué pedir a continuación hasta recibir la respuesta anterior. El café la
tiene, porque las claves salen del catálogo y las unidades salen de las claves.

Las dos celdas de agente que siguen la fuerzan a propósito, y las dos llegan a
cuatro vueltas por caminos distintos.

In [9]:
# Dependencia real: escribir, y despues recalcular sobre el resultado.
# Da 4 vueltas y 12 llamadas: la escritura, el catalogo, los diez stocks
# juntos, y la respuesta.
requests.post(f"{p3.BACKEND}/api/admin/reiniciar", timeout=20)
p3.correr("Pon la miel en 100 unidades y luego dime cuál es ahora el producto con más stock")


  --- vuelta 1 ---
  pide: actualizar_existencias({'sku': 'MIEL-500', 'existencias': 100})
  resultado: {'sku': 'MIEL-500', 'nombre': 'Miel de abeja 500g', 'existencias': 100, 'precio': 120.0, 'categoria': 'endulzante'}

  --- vuelta 2 ---
  pide: consultar_catalogo({})
  resultado: {'productos': [{'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'precio': 180.0, 'categoria': 'cafe'}, {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'precio': 95.0, 'categoria': 'cafe'}, {'sku': 'CAFE-GRANO-1K', 'nombre': 'Café en grano 1kg', 'precio': 340.0, 'categoria': 'cafe'}, {'sku': 'TE-200', 'nombre': 'Té verde 200g', 'precio': 95.0, 'categoria': 'te'}, {'sku': 'TE-MANZ-100', 'nombre': 'Té de manzanilla 100g', 'precio': 62.0, 'categoria': 'te'}, {'sku': 'AZUCAR-1K', 'nombre': 'Azúcar 1kg', 'precio': 38.0, 'categoria': 'endulzante'}, {'sku': 'MIEL-500', 'nombre': 'Miel de abeja 500g', 'precio': 120.0, 'categoria': 'endulzante'}, {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'precio': 29.0, 'c

'El producto con más stock es la **Miel de abeja 500g** con 100 unidades.'

### Varias tools en la misma vuelta

La celda siguiente llega a **cuatro vueltas y quince llamadas**, y lo que importa
es cómo se reparten:

| vuelta | llamadas |
|---|---|
| 1 | `consultar_catalogo({})` |
| 2 | **diez** `consultar_stock`, uno por producto, juntas |
| 3 | **cuatro** `actualizar_existencias`, juntas |
| 4 | responde |

**Las vueltas 2 y 3 son el `for pedido in pedidos` del bucle.** Cuando el modelo
no necesita el resultado de una llamada para pedir la siguiente, las manda todas
juntas y tu programa las ejecuta en la misma vuelta. Una vuelta puede traer
varias peticiones.

Y el reparto no es casual: tiene que leer los diez stocks **antes** de saber
cuáles están por debajo de diez. Por eso hay dos vueltas de llamadas y no una.

> El bucle que ejecuta esto es el del paso 2, sin una línea de cambio. Ahí
> también se ejercita: el notebook 02 llega a diez `consultar_stock` en una sola
> vuelta cuando se le pide el valor de todo el inventario.

In [10]:
requests.post(f"{p3.BACKEND}/api/admin/reiniciar", timeout=20)
p3.correr("Sube a 20 unidades todos los productos que tengan menos de 10")

print()
tabla()


  --- vuelta 1 ---
  pide: consultar_catalogo({})
  resultado: {'productos': [{'sku': 'CAFE-500', 'nombre': 'Café molido 500g', 'precio': 180.0, 'categoria': 'cafe'}, {'sku': 'CAFE-250', 'nombre': 'Café molido 250g', 'precio': 95.0, 'categoria': 'cafe'}, {'sku': 'CAFE-GRANO-1K', 'nombre': 'Café en grano 1kg', 'precio': 340.0, 'categoria': 'cafe'}, {'sku': 'TE-200', 'nombre': 'Té verde 200g', 'precio': 95.0, 'categoria': 'te'}, {'sku': 'TE-MANZ-100', 'nombre': 'Té de manzanilla 100g', 'precio': 62.0, 'categoria': 'te'}, {'sku': 'AZUCAR-1K', 'nombre': 'Azúcar 1kg', 'precio': 38.0, 'categoria': 'endulzante'}, {'sku': 'MIEL-500', 'nombre': 'Miel de abeja 500g', 'precio': 120.0, 'categoria': 'endulzante'}, {'sku': 'LECHE-1L', 'nombre': 'Leche entera 1L', 'precio': 29.0, 'categoria': 'lacteo'}, {'sku': 'LECHE-DESL-1L', 'nombre': 'Leche deslactosada 1L', 'precio': 34.0, 'categoria': 'lacteo'}, {'sku': 'VASO-360', 'nombre': 'Vaso desechable 360ml (50 pz)', 'precio': 85.0, 'categoria': 'desech

### Una decisión de diseño que no se ve, pero manda

Mira lo que devolvió `consultar_catalogo` en las celdas de la sección 1: clave,
nombre, precio y categoría. **Las existencias no están.** Para un número de
unidades hace falta una segunda llamada, y eso es lo que convierte casi cada
pregunta en dos vueltas o más.

Esa es una decisión tuya, no del modelo, y está medida. El 2026-11-15 se
probaron tres formas del catálogo contra el servidor del Tec, cinco preguntas
en cada una:

| el catálogo devuelve… | qué pasa |
|---|---|
| **todo**, con existencias | una llamada resuelve casi cualquier pregunta. El bucle no pasa de dos vueltas y el `for pedido in pedidos` no se ejercita en ningún lado |
| **clave, nombre y precio** | el modelo encadena: más vueltas, más latencia, y cada número sale de donde vive |
| **solo clave y nombre** | encadena igual, pero pidió dos de cinco productos y contestó como si fueran los cinco |

Se eligió la de en medio. **Devolver de menos no vuelve al modelo más
trabajador**, y devolver de más le quita el trabajo por completo. El detalle de
las tres está en [`planeacion/hallazgos.md`](../planeacion/hallazgos.md).

> **Lo que cuesta devolver de menos es la respuesta de la sección 1.** Sin
> existencias en el catálogo, *"¿hay té verde?"* se contesta con *"sí, hay"*
> sobre un producto en cero. El dato estaba a una llamada y nada obligó a
> hacerla.

Lo que cuesta devolver todo es contexto. **La ventana por omisión de Ollama son
4 096 tokens**, y los diez productos completos no son gratis ahí. El servidor
del Tec corre mucho más ancho, 262 144 tokens según su `/api/ps`, así que en el
taller no aprieta; en una laptop con la omisión, sí.

Cuánto cabe de verdad en esa ventana, y qué pasa al pasarse, está medido en la
sección 2 del [notebook 01](01_del_prompt_al_agente.ipynb).

## Dejar el inventario como estaba

La celda siguiente devuelve el inventario a su estado inicial. Córrela al
terminar, o el notebook siguiente arranca con los números que dejó este.

In [11]:
r = requests.post(f"{p3.BACKEND}/api/admin/reiniciar", timeout=20).json()
print(r)
tabla()

{'estado': 'reiniciado', 'productos': 10}
  SKU             producto                        exist.  precio  categoría
  --------------- ------------------------------ ------- -------  -----------
  CAFE-500        Café molido 500g                    17     180  cafe
  CAFE-250        Café molido 250g                    34      95  cafe
  CAFE-GRANO-1K   Café en grano 1kg                    8     340  cafe
  TE-200          Té verde 200g                        0      95  te  <- en cero
  TE-MANZ-100     Té de manzanilla 100g               23      62  te
  AZUCAR-1K       Azúcar 1kg                          42      38  endulzante
  MIEL-500        Miel de abeja 500g                  11     120  endulzante
  LECHE-1L        Leche entera 1L                      3      29  lacteo
  LECHE-DESL-1L   Leche deslactosada 1L               19      34  lacteo
  VASO-360        Vaso desechable 360ml (50 pz)        7      85  desechable

  10 productos


## Tu turno

Los cuatro `# TODO (taller)` del final de
[`src/paso3_tools_http/agente.py`](../src/paso3_tools_http/agente.py).

El TODO 4 es el que cierra el bloque. Compara los dos archivos con

```
git diff --no-index src/paso2_herramienta/agente.py src/paso3_tools_http/agente.py
```

y cuenta cuántas líneas **del bucle** cambiaron.

---

### Lo que queda sin resolver, y es el paso 4

`actualizar_existencias` modifica el sistema y **nadie le pidió permiso a
nadie**. En este inventario no pasa nada grave; cuando la tool sea
`borrar_producto`, sí.

Esa es la pregunta con la que abre el paso 4: **¿qué acciones merecen una
pausa?** Si se confirman todas, nadie lee las confirmaciones. Si no se confirma
ninguna, el agente borra algo un martes.